In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


# SAR fundamentals

## Bands and wavelengths

| Frequency Band | Ka | Ku | X | C | S | L | P |
|---|---|---|---|---|---|---|---|
| **Frequency [GHz]** | 40–25 | 17.6–12 | 12–7.5 | 7.5–3.75 | 3.75–2 | 2–1 | 0.5–0.25 |
| **Wavelength [cm]** | 0.75–1.2 | 1.7–2.5 | 2.5–4 | 4–8 | 8–15 | 15–30 | 60–120 |

In [3]:
satellites = {
    "TerraSAR-X": {"band": "X", "wavelength_cm": 3.1},
    "Sentinel-1": {"band": "C", "wavelength_cm": 5.6},
    "RadarSat(RCM)": {"band": "C", "wavelength_cm": 5.6},
    "ALOS-4":     {"band": "L", "wavelength_cm": 24},
    "BIOMASS":    {"band": "P", "wavelength_cm": 70},
    "NISAR":      {"band": "L", "wavelength_cm": 24},
}



In [4]:
#browse the satelite dictionary
print("Satelites: ", [satellite for satellite in satellites])
print(satellites["TerraSAR-X"])

Satelites:  ['TerraSAR-X', 'Sentinel-1', 'RadarSat(RCM)', 'ALOS-4', 'BIOMASS', 'NISAR']
{'band': 'X', 'wavelength_cm': 3.1}


## Repeat vs Revist

**Repeat cycle:** return of the spacecraft to the same relative orbit

**revisit:** time until the same $m^2$ on earth surface is imaged again

## Directions

Radar systems will provide a 2-D Reflectivity map of an imaged area. The flight direction is denoted as azimuth and the line-of-sight as slant range direction. 

## Antenna patterns

Azimuth resolution is closely relates to the dimensions of the radar antenna. A small antenna with a broad beamwidth results in a broader azimuth bandwidth.
The Antenna Pattern (AP) is a function of the off-centre beam angle (squint angle) $\phi$ in both range and azimuth direction. 

$$
\text{AP}= sinc^2(\frac{D_a}{\lambda_r}\phi_a)sinc^2(\frac{L_a}{\lambda_a}\phi_a)
$$



```{figure} ../../figures/antenna_pattern.png
---
height: 300px
name: AAP

---
```

Below is an interactive plot of the antenna pattern in the azimuth direction as a function of the beam angle (which can also be translated to the doppler frequency). You can explore the relationship between the antenna length and the azimuth antenna pattern. 





In [26]:
def doppler_frequency(velocity, wavelength, angle):
    return (2 * velocity * np.sin(angle)) / wavelength

def one_way_AAP_general(l_a, phi, wl):
    return np.sinc(l_a * np.sin(phi) / wl)**2

def plot_AAP(antenna_length):
    f = 5.331e9
    wl = 3e8 / f
    v = 7.45 * 1000

    angle_deg = np.linspace(-0.8, 0.8, 2000)
    angle_rad = np.radians(angle_deg)

    AAP = one_way_AAP_general(antenna_length, angle_rad, wl)**2

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(angle_deg, 10 * np.log10(AAP + 1e-10), 'b-', label=f'AAP (L={antenna_length} m)')
    ax1.set_xlabel('Squint Angle (degrees)', fontsize=12)
    ax1.set_ylabel('Two-Way Gain (dB)', fontsize=12)
    ax1.set_title('Two-Way Azimuth Antenna Pattern', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twiny()
    ax2.set_xlim(ax1.get_xlim())
    angle_limits_rad = np.radians(np.array(ax1.get_xlim()))
    doppler_limits = doppler_frequency(v, wl, angle_limits_rad)
    ax2.set_xlim(doppler_limits)
    ax2.set_xlabel('Doppler Frequency (Hz)', fontsize=12)

    plt.tight_layout()
    plt.show()

# --- Widget ---
antenna_slider = widgets.FloatSlider(
    value=10, min=6, max=14, step=0.1,
    description='Antenna length (m):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px'),
    readout_format='.1f'
)

out = widgets.interactive_output(plot_AAP, {'antenna_length': antenna_slider})
display(antenna_slider, out)

FloatSlider(value=10.0, description='Antenna length (m):', layout=Layout(width='500px'), max=14.0, min=6.0, re…

Output()

Beam stearing

```{figure} ../../figures/beamsteering.png
---
height: 300px
name: AAP

---
```

In [ ]:

def interf_simulation(N_elements, dx, phase_offset):
    N = 512
    wavelength = 50
    twopi = 2 * np.pi
    k = twopi / wavelength

    center = [0, 127]
    source_loc = np.array([
        [center[0]] * N_elements,
        center[1] + np.arange(0, dx * N_elements, dx)
    ]).T.astype(int)

    comp = np.ones((N, N), dtype=complex)
    ampl = np.zeros((N, N))

    for i in range(N_elements):
        rows = np.arange(N).reshape(1, -1) - source_loc[i, 1]
        cols = np.arange(N).reshape(-1, 1) - source_loc[i, 0]
        range_ = np.sqrt(rows**2 + cols**2)

        phase_abs = k * range_ + phase_offset * (i + 1)
        phase = np.mod((phase_abs + np.pi), twopi) - np.pi

        comp *= np.conj(np.exp(1j * phase))
        ampl += np.real(np.conj(np.exp(1j * phase)))

    DB_map = 10 * np.log10(np.abs(ampl)**2 + 1e-10)

    # --- Row 1: Phase and Amplitude ---
    fig1, axes1 = plt.subplots(1, 2, figsize=(16, 6))

    im1 = axes1[0].imshow(np.angle(comp), cmap='gray')
    axes1[0].set_title(f'Phase of signal; # sources: {N_elements}')
    plt.colorbar(im1, ax=axes1[0])
    axes1[0].axis('image')
    axes1[0].plot(source_loc[:, 1], source_loc[:, 0], 'ro')

    im2 = axes1[1].imshow(np.abs(ampl), cmap='gray')
    axes1[1].set_title(f'Amplitude; # sources: {N_elements}')
    plt.colorbar(im2, ax=axes1[1])
    axes1[1].axis('image')
    axes1[1].plot(source_loc[:, 1], source_loc[:, 0], 'ro')

    plt.tight_layout()
    plt.show()

    # --- Row 2: dB map and Power profile ---
    fig2, axes2 = plt.subplots(1, 2, figsize=(16, 6))

    im3 = axes2[0].imshow(DB_map, cmap='jet', vmin=-10, vmax=np.max(DB_map))
    axes2[0].set_title('Amplitude in dB')
    plt.colorbar(im3, ax=axes2[0])
    axes2[0].axis('image')

    axes2[1].plot(DB_map[N//2, :], 'k-o')
    axes2[1].set_title('Power/Intensity at the ground [dB]')
    axes2[1].grid()

    plt.tight_layout()
    plt.show()

# --- Widgets ---
n_slider = widgets.IntSlider(
    value=1, min=1, max=15, step=1,
    description='N elements:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

dx_slider = widgets.IntSlider(
    value=10, min=2, max=30, step=1,
    description='dx (spacing):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

phase_slider = widgets.FloatSlider(
    value=0, min=0.0, max=np.pi, step=0.01,
    description='Phase offset:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
    readout_format='.2f'
)

ui = widgets.VBox([n_slider, dx_slider, phase_slider])
out = widgets.interactive_output(
    interf_simulation,
    {'N_elements': n_slider, 'dx': dx_slider, 'phase_offset': phase_slider}
)

display(ui, out)


Output()

## Resolution
- azimuth resolution: syntethetic antenna, physical antenna pic 


**Azimuth resolution** $\delta_a$ is the smallest seperation between two point targets that can be detected by radar

For simple side looking radars the azimuth antenna beamwidth $\Theta$ has the following relation with wavelength $\lambda$ and an azimuth antenna length $d_a$:

$$
\Theta= \frac{\lambda}{d_a}
$$

The azimuth resolution $\delta_a$ will increase with range distance $r_0$:

$$
\delta_a = \frac{\lambda}{d_a}\cdot r_0 = \Theta \cdot r_0
$$





```{figure} ../../figures/along_track_resolution.png
---
height: 300px
name: AAP

---
```

```{figure} ../../figures/Azimuth_resolution.png
---
height: 300px
name: AAP

---
```


For **SAR** the azimuth resolution is equal to half the azimuth antenna length and is independent of the range distance.


$$
\delta_a = \frac{d_a}{2}
$$

In [24]:
def rar_resolution(wl,d_a, r0):
    """
    Compute the azimuth resolution of a Real Apperture Radar system.
    
    Parameters:
    wl (float): Wavelength of the radar signal (in meters).
    d_a (float): Diameter of the antenna (in meters).
    r0 (float): Range to the target (in meters).
    
    Returns:
    float: Azimuth resolution (in meters).
    """
    return (wl/d_a) * r0

def sar_resolution(d_a):
    """
    Compute the azimuth resolution of a Synthetic Aperture Radar system.
    
    Parameters:
    wl (float): Wavelength of the radar signal (in meters).
    d_a (float): antenna length (in meters).
    
    Returns:
    float: Azimuth resolution (in meters).
    """
    return d_a/2

#play around with azimuth resolution for different wavelengths and antenna sizes
bands = {'X':3, 'C':5, 'L':20, 'P':100} #cm
antenna_sizes = {'1 m': 1, '5 m': 5, '10 m': 10, '20 m': 20} #m

In [28]:
#play around with azimuth resolution for different wavelengths and antenna sizes
print(f"inspect bands dictionary: {bands} with wavelengths in cm")
print(f"inspect antenna_sizes set: {antenna_sizes}")
print("\n")

#pick your parameters to compute azimuth resolution for RAR and SAR systems
r0 = 1000e3 #m
band = bands['C'] #cm. 
antenna_size = antenna_sizes['10 m']

#azimuth resolution for RAR and SAR systems
print(f"Azimuth resolution for RAR at {r0/1e3} km range with a {antenna_size} m antenna:")
print(f"{rar_resolution(band*1e-2, antenna_size, r0):.2f} m")

#azimuth resolution for SAR
print(f"Azimuth resolution for SAR for an antenna of length {antenna_sizes['10 m']} m:")
print(f"{sar_resolution(antenna_sizes['10 m']):.2f} m")

inspect bands dictionary: {'X': 3, 'C': 5, 'L': 20, 'P': 100} with wavelengths in cm
inspect antenna_sizes set: {'1 m': 1, '5 m': 5, '10 m': 10, '20 m': 20}


Azimuth resolution for RAR at 1000.0 km range with a 10 m antenna:
5000.00 m
Azimuth resolution for SAR for an antenna of length 10 m:
5.00 m


## Range compression
